<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-25</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>환경 설정과 LLM 한계(Environment Setup & LLM Limitations)</strong>에 대해 학습합니다.
>LangChain ChatUpstage를 초기화하고, LLM의 환각(Hallucination) 문제를 직접 학습해봅시다.

</br>

<div style="text-align:center">

</div>

In [10]:
# TODO 0: 환경변수를 로드해봅시다.

from dotenv import load_dotenv

load_dotenv()
print("환경변수 로드 완료")

환경변수 로드 완료


</br>

# LLM 초기화 (LLM Initialization)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ChatUpstage</mark>를 사용하여 LLM을 초기화하고 메시지를 전달합니다.

> LLM은 강력하지만, 실무에서 그대로 사용하기엔 분명한 한계가 있습니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">할루시네이션(Hallucination)</mark>으로 학습 데이터에 없는 내용을 그럴듯하게 지어내고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">최신 정보 부재</mark>로 학습 시점 이후의 사건이나 수치를 알지 못하며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">외부 도구 사용 불가</mark>로 인터넷 검색이나 파일 읽기 등의 실행 능력이 없고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토큰 제한</mark>으로 매우 긴 문서를 통째로 입력할 수 없습니다. 이 한계들을 이해하는 것이 RAG와 Agent 패턴을 배우는 출발점입니다.</br></br>
> 이번 실습에서는 언어 모델이 토큰 단위로 다음 단어를 예측하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM 기본 개념</mark>을 전제로, HTTP 요청/응답 구조와 API 키 인증 방식 등 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">API 호출</mark> 기초, 그리고 시스템 메시지와 사용자 메시지를 구분하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">프롬프팅</mark> 지식을 활용합니다.

In [11]:
# TODO 1: LLM 객체를 solar-pro3 모델, temperature=0으로 초기화하고, 사용자 메시지 "안녕하세요"를 전달하여 응답과 모델명을 출력해봅시다.

from langchain_upstage import ChatUpstage
from langchain_core.messages import HumanMessage

# LLM 초기화
llm = ChatUpstage(
    model="solar-pro3",
    temperature=0
)

# 메시지 전달
response = llm.invoke([HumanMessage(content="안녕하세요")])
print(response.content)
print(f"모델: {response.response_metadata['model_name']}")

안녕하세요! 😊  
무엇을 도와드릴까요? 궁금한 점이나 이야기하고 싶은 주제가 있으면 편하게 말씀해 주세요.
모델: solar-pro3-260323


</br>

## HumanMessage와 invoke
> LangChain에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">HumanMessage 객체</mark>로 사용자 입력을 구조화하고, `invoke()`로 LLM에 전달합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>ChatUpstage</code></td><td>Upstage 모델 래퍼 (LangChain)</td></tr>
    <tr><td style="text-align:center"><code>HumanMessage</code></td><td>사용자 메시지 객체</td></tr>
    <tr><td style="text-align:center"><code>invoke()</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단일 입력에 대한 응답</mark> 생성</td></tr>
    <tr><td style="text-align:center"><code>temperature</code></td><td>0: 결정적, 1: 창의적</td></tr>
  </tbody>
</table>

</br>

# LLM의 한계: 환각 (Hallucination)
> LLM은 학습 데이터에 없는 내용을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">그럴듯하게 지어내는</mark> 현상이 발생합니다.

In [13]:
# TODO 2: 사용자 메시지로 "예스24 총알배송의 구체적인 배송 기준과 혜택을 알려줘"를 LLM에 전달하고, 환각 응답을 확인해봅시다.

response = llm.invoke([
    HumanMessage(content="예스24 총알배송의 구체적인 배송 기준과 혜택을 알려줘")
])
print(response.content)

2025년 3분기(7월 ~ 9월) 삼성전자의 매출액은 아직 공식적으로 발표되지 않았습니다.  
삼성전자는 매 분기 종료 후 약 4~6주 뒤에 **실적 발표**(Earnings Release)를 통해 매출액, 영업이익, 순이익 등을 공개합니다.  

- **예상 발표 시점**: 2025년 10월 중순 ~ 말경  
- **확인 방법**  
  1. **삼성전자 IR(Investor Relations) 홈페이지** – “실적 발표” 섹션에 최신 분기 보고서가 업로드됩니다.  
  2. **한국거래소(KRX) 공시** – “공시정보 → 기업별 공시 → 삼성전자”에서 확인할 수 있습니다.  
  3. **주요 금융·경제 뉴스** – Bloomberg, Reuters, 한국경제, 매일경제, 연합뉴스 등에서 실적 발표 직후 보도됩니다.  

현재 시점(2026‑03‑24)에서는 2025년 3분기 실적이 이미 발표되었을 가능성이 높습니다. 최신 정보를 확인하려면 위 경로를 통해 **2025년 10월 ~ 11월**에 공개된 “2025년 3분기 실적 보고서”를 찾아보시면 됩니다.  

추가로 필요하신 정보(예: 전년 동기 대비 성장률, 부문별 매출, 영업이익 등)가 있으면 알려 주세요. 그에 맞춰 최신 데이터를 찾아 정리해 드리겠습니다.


💡위 출력은 LLM이 생성한 **허구**입니다. 실제 데이터와 다를 수 있습니다.

</br>

## 환각의 원인과 해결

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">원인</th>
      <th>설명</th>
      <th>해결 방안</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">학습 데이터 한계</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">최신 정보 부재</mark></td><td>RAG (외부 자료 주입)</td></tr>
    <tr><td style="text-align:center">과잉 일반화</td><td>패턴 기반 생성</td><td>출처 명시 요구</td></tr>
    <tr><td style="text-align:center">확신 편향</td><td>"모른다" 대신 생성</td><td>temperature 조절</td></tr>
  </tbody>
</table>

💡왜 RAG가 필요한가?
> LLM은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 시점 이후의 정보</mark>를 알 수 없습니다.
> 외부 자료를 검색하여 컨텍스트로 제공하면 환각을 크게 줄일 수 있습니다.

💡temperature=0이어도 환각 발생
> temperature=0은 출력의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재현성</mark>을 높일 뿐, 환각 자체를 방지하지는 않습니다.